# EN.737 — GPT-2 fine-tuning for German → English
**M1 and Q1(a): full fine-tuning versus rank-8 LoRA**

Train the same pretrained GPT-2 small checkpoint on the team's IWSLT 2017 TED parallel text. Compare full updates against LoRA on the fused QKV projection (`c_attn`) at an identical supervised-token budget. No M2 training is done here.

### Run instructions
1. In Colab select **Runtime → Change runtime type → GPU** (a T4 is enough). Run the setup cell; if Colab requests a restart, restart and rerun.
2. Obtain the `shared_splits/` folder (or ZIP containing it) from the M2 run. It contains `train.tsv`, `validation.tsv`, `test.tsv`, with `source`, `target`, and optionally `id`. Upload the ZIP or put the files in Drive. The project profile intentionally refuses to silently resample data. See [team repository](https://github.com/NimaChelongar/en737-translation).
3. Edit settings, especially `DATA_DIR`, `OUTPUT_ROOT`, and the prediction **before training**. Run the notebook top to bottom. `PROFILE="smoke"` tests the code on toy pairs, not for reporting. Project defaults run both methods for three paired seeds (17, 42, 43), as T7 requires repeated runs; each completed run is saved, so a disconnected session resumes where it stopped. Completed runs are reused when configuration, model revision and split hashes match.
4. Results go to a versioned experiment folder. The final cell creates a report ZIP (excluding model weights). Keep the merged best checkpoint for Notebook 2.

The report uses the same zero-shot prompt, greedy newline/EOS stopping and 128-token cap as M0. Training supervises the English continuation and EOS; reported reference CE excludes EOS and counts UTF-8 bytes of the continuation including its initial space, as in M0. No test data is used to choose checkpoints or hyperparameters. The primary Q1 comparison uses **final checkpoints** at identical exposure; a validation-selected checkpoint is also evaluated for M1 and the demo.

In [ ]:
# Colab dependencies; leave the installed GPU build of PyTorch alone.
import sys, subprocess, importlib.metadata as im
pins = {"transformers":"4.51.3", "peft":"0.15.2", "sacrebleu":"2.5.1", "accelerate":"1.6.0"}
missing = []
for package, version in pins.items():
    try: installed = im.version(package)
    except im.PackageNotFoundError: installed = None
    if installed != version: missing.append(f"{package}=={version}")
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])

## 1. Configuration and preregistered prediction
Default hyperparameters are proposed, not claimed to be optimal. The separate learning rates are fixed before training. The comparison is conditional on this configuration; it does not prove the best attainable LoRA/full performance. Keep the token budget and batch order matched.

In [ ]:
import os, math, json, random, hashlib, time, gc, re, csv, zipfile, shutil, platform
from pathlib import Path
from datetime import datetime, timezone
from contextlib import nullcontext
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F
from transformers import AutoModelForCausalLM, AutoTokenizer, get_linear_schedule_with_warmup
from peft import LoraConfig, TaskType, get_peft_model, PeftModel
from huggingface_hub import HfApi
from sacrebleu.metrics import BLEU, CHRF
from IPython.display import display
from tqdm.auto import tqdm

PROFILE = "project"          # "project" or "smoke"
USE_DRIVE = "auto"           # "auto": mount Drive in Colab, fall back to runtime disk if mounting fails (e.g. Colab VS Code extension); True: require Drive; False: never mount
DATA_DIR = "/content/team_data" if Path("/content").exists() else "./team_data"
DATA_ZIP = ""                 # optional path to ZIP from M2; auto-finds shared_splits
OUTPUT_ROOT = "/content/en737_m1_runs" if Path("/content").exists() else "./en737_m1_runs"
if USE_DRIVE == "auto": USE_DRIVE = None if Path("/content").exists() else False
assert USE_DRIVE in {None, True, False}, "USE_DRIVE must be 'auto', True or False"
if USE_DRIVE is not False:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        OUTPUT_ROOT = "/content/drive/MyDrive/en737_m1_runs"
        USE_DRIVE = True
        # Change DATA_DIR / DATA_ZIP above to your actual Drive path if needed.
    except Exception as e:
        if USE_DRIVE: raise
        USE_DRIVE = False
        print(f"Drive not mounted ({type(e).__name__}: {e}). Saving to {OUTPUT_ROOT} on the runtime disk; "
              "download the report zip before the runtime disconnects.")

MODEL_ID = "openai-community/gpt2"
MODEL_REVISION = None          # ideally use the same commit as M0; None resolves and records current SHA
SEEDS = [17, 42, 43]           # paired seeds for variability (T7); completed runs are reused on rerun
METHODS = ["full", "lora"]
CFG = dict(epochs=3, micro_batch=4, accumulation=8, eval_batch=8, gen_batch=8,
           full_lr=5e-5, lora_lr=3e-4, weight_decay=0.01, warmup_fraction=0.05,
           grad_clip=1.0, lora_rank=8, lora_alpha=16, lora_dropout=0.0,
           context=1024, max_new_tokens=128, log_every=50, bootstrap_samples=1000,
           precision="auto", length_bucket_window=128)
# Set before scoring/training; recorded with the experiment.
PREDICTION = {"hypothesis": "Rank-8 LoRA is competitive with full fine-tuning at a matched target-token budget.",
              "primary_metric": "final checkpoint test corpus chrF (LoRA minus full)",
              "minimum_delta_chrf": -2.0,
              "secondary_prediction": "LoRA uses less peak allocated training GPU memory than full fine-tuning."}
if PROFILE == "smoke":
    CFG.update(epochs=1, micro_batch=2, accumulation=2, eval_batch=2, gen_batch=2,
               max_new_tokens=12, log_every=1, bootstrap_samples=20)
    SEEDS = [17]
assert PROFILE in {"project", "smoke"} and len(set(SEEDS)) == len(SEEDS) and SEEDS
assert set(METHODS) <= {"full", "lora"} and METHODS
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if PROFILE == "project" and DEVICE.type != "cuda":
    raise RuntimeError("Select a GPU runtime before the project run.")
if DEVICE.type == "cpu": torch.set_num_threads(min(4, torch.get_num_threads()))
AMP_DTYPE = (torch.bfloat16 if DEVICE.type == "cuda" and torch.cuda.get_device_capability(DEVICE)[0] >= 8 else torch.float16)
AMP = DEVICE.type == "cuda" and CFG["precision"] != "fp32"
if DEVICE.type == "cuda" and CFG["precision"] == "fp16": AMP_DTYPE = torch.float16
if CFG["precision"] == "bf16":
    assert DEVICE.type == "cuda" and torch.cuda.is_bf16_supported()
    AMP_DTYPE = torch.bfloat16
assert CFG["precision"] in {"auto", "fp32", "fp16", "bf16"}
print("Device:", DEVICE, torch.cuda.get_device_name() if DEVICE.type == "cuda" else "CPU",
      "| compute dtype:", AMP_DTYPE if AMP else torch.float32)

def dump(path, obj):
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(obj, indent=2, ensure_ascii=False, default=str), encoding="utf-8")
def sha(path): return hashlib.sha256(Path(path).read_bytes()).hexdigest()
def seed_all(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if DEVICE.type == "cuda": torch.cuda.manual_seed_all(seed)
def sync():
    if DEVICE.type == "cuda": torch.cuda.synchronize()
def clear():
    gc.collect()
    if DEVICE.type == "cuda": torch.cuda.empty_cache()
def amp_context():
    return torch.autocast(device_type="cuda", dtype=AMP_DTYPE) if AMP else nullcontext()

## 2. Shared splits and reproducibility
The M2 export may contain fewer than 20,000 training rows because of its context filter. We preserve the actual exported cohort and report its size rather than force 20,000. No reference is truncated. If a sequence does not fit GPT-2, stop and coordinate a common evaluation cohort with the team.

In [ ]:
SPLITS = ("train", "validation", "test")
if PROFILE == "smoke":
    rows = {}
    for split, n in zip(SPLITS, [16, 4, 4]):
        rows[split] = [dict(id=f"{split}-{i}", source=f"Das ist Nummer {split} {i}.",
                            target=f"This is number {split} {i}.") for i in range(n)]
    hashes = {s:hashlib.sha256(json.dumps(rows[s], sort_keys=True).encode()).hexdigest() for s in SPLITS}
else:
    folder = Path(DATA_DIR).expanduser()
    if DATA_ZIP:
        folder.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(DATA_ZIP) as z:
            for split in SPLITS:
                candidates = [n for n in z.namelist() if n.endswith(f"shared_splits/{split}.tsv") or n == f"{split}.tsv"]
                if len(candidates) != 1: raise ValueError(f"Expected one {split}.tsv, found {candidates}")
                (folder / f"{split}.tsv").write_bytes(z.read(candidates[0]))
    if not all((folder / f"{s}.tsv").exists() for s in SPLITS):
        candidates = [p for p in folder.rglob("shared_splits") if all((p / f"{s}.tsv").exists() for s in SPLITS)]
        if len(candidates) == 1: folder = candidates[0]
    if not all((folder / f"{s}.tsv").exists() for s in SPLITS):
        raise FileNotFoundError(f"Set DATA_DIR to the M2 shared_splits folder or DATA_ZIP to its archive. Missing files in {folder}.")
    rows, hashes = {}, {}
    for split in SPLITS:
        path = folder / f"{split}.tsv"
        hashes[split] = sha(path)
        with path.open(encoding="utf-8-sig", newline="") as f:
            reader = csv.DictReader(f, delimiter="\t")
            assert {"source", "target"}.issubset(reader.fieldnames or []), path
            rows[split] = [dict(id=str(r.get("id") or i), source=r["source"], target=r["target"])
                           for i,r in enumerate(reader)]
for split in SPLITS:
    assert rows[split] and len({r['id'] for r in rows[split]}) == len(rows[split]), split
    assert all(r['source'].strip() and r['target'].strip() and '\n' not in r['target'] for r in rows[split]), split
    print(split, len(rows[split]), hashes[split][:16])
for i,s in enumerate(SPLITS):
    for t in SPLITS[i+1:]:
        assert not ({r['source'] for r in rows[s]} & {r['source'] for r in rows[t]}), f"Source leakage {s}/{t}"
if PROFILE == "project" and (len(rows['validation']) != 512 or len(rows['test']) != 512):
    print("WARNING: shared validation/test counts differ from the report's 512; verify with teammates.")

# Resolve a concrete revision once; all runs use it, not a moving branch.
REVISION = MODEL_REVISION or HfApi().model_info(MODEL_ID).sha
assert REVISION, "Set MODEL_REVISION to the checkpoint commit if API resolution fails."
import importlib.metadata as im
versions = {p: im.version(p) for p in ['transformers', 'peft', 'sacrebleu', 'accelerate', 'torch', 'tokenizers', 'huggingface-hub', 'numpy', 'pandas']}
experiment = dict(profile=PROFILE, model=MODEL_ID, revision=REVISION, cfg=CFG, prediction=PREDICTION,
                  split_sha256=hashes, split_sizes={s:len(rows[s]) for s in SPLITS},
                  software=versions, hardware=dict(device=str(DEVICE),
                  precision=str(AMP_DTYPE if AMP else torch.float32), python=platform.python_version()))
# The GPU model is recorded per run (config.json, complete.json) but kept out of the experiment ID, so a
# Colab reconnect onto a different GPU of the same precision resumes this experiment instead of restarting it.
GPU_NAME = torch.cuda.get_device_name() if DEVICE.type=='cuda' else None
EXPERIMENT_ID = hashlib.sha256(json.dumps(experiment, sort_keys=True).encode()).hexdigest()[:12]
OUT = Path(OUTPUT_ROOT) / f"{PROFILE}-{EXPERIMENT_ID}"
OUT.mkdir(parents=True, exist_ok=True)
if not (OUT / 'preregistration.json').exists():
    dump(OUT / 'preregistration.json', dict(recorded_at=datetime.now(timezone.utc).isoformat(), gpu=GPU_NAME, **experiment))
print("Experiment:", OUT, "| revision:", REVISION)

## 3. Tokenization and target masking
For a pair $(x,y)$, tokenize `German: {source}\nEnglish:` and the continuation ` {target}` separately, then append EOS. Training labels for prompt and padding positions are `-100` (ignored). The first English token is predicted from the final prompt position. Training includes EOS, while validation/test CE measures only reference tokens for compatibility with M0. The numerator is total negative log likelihood, not an average of batch means.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, revision=REVISION, use_fast=True)
EOS = tokenizer.eos_token_id
PAD = EOS
# Stop when any decoded vocabulary item contains a newline, or EOS (as in M0).
STOP_IDS = sorted({EOS} | {i for i in range(len(tokenizer)) if '\n' in tokenizer.decode([i], clean_up_tokenization_spaces=False)})
STOP_SET = set(STOP_IDS)
def prompt(source): return f"German: {source}\nEnglish:"
def encode_text(s): return tokenizer.encode(s, add_special_tokens=False)
def encode_row(r):
    p, t = encode_text(prompt(r['source'])), encode_text(' '+r['target'])
    if EOS in p or EOS in t: raise ValueError(f"Literal EOS in row {r['id']}; coordinate cleaning with team.")
    if not t or len(p)+len(t)+1 > CFG['context'] or len(p)+CFG['max_new_tokens'] > CFG['context']:
        raise ValueError(f"Row {r['id']} exceeds context; coordinate shared cohort rather than truncate.")
    return dict(**r, p=p, t=t, target_bytes=len((' '+r['target']).encode('utf-8')))
encoded = {s:[encode_row(r) for r in tqdm(rows[s], desc=f'Tokenize {s}')] for s in SPLITS}

def collate(examples, train=False):
    targets = [r['t']+([EOS] if train else []) for r in examples]
    lengths = [len(r['p'])+len(t) for r,t in zip(examples,targets)]
    width = max(lengths)
    ids = torch.full((len(examples),width), PAD, dtype=torch.long)
    mask = torch.zeros_like(ids)
    labels = torch.full_like(ids,-100)
    for i,(r,t,n) in enumerate(zip(examples, targets, lengths)):
        k=len(r['p']); ids[i,:n]=torch.tensor(r['p']+t); mask[i,:n]=1; labels[i,k:n]=torch.tensor(t)
    return ids, mask, labels

def batches(items, size):
    for i in range(0,len(items),size): yield items[i:i+size]

def epoch_batches(seed, epoch):
    # Same shuffle + local length bucketing for both methods. Every row appears once per epoch.
    indices = list(range(len(encoded['train'])))
    random.Random(seed + 10007*epoch).shuffle(indices)
    ordered=[]
    for window in batches(indices,CFG['length_bucket_window']):
        ordered.extend(sorted(window,key=lambda i:len(encoded['train'][i]['p'])+len(encoded['train'][i]['t'])))
    return [[encoded['train'][i] for i in b] for b in batches(ordered,CFG['micro_batch'])]

def supervised_count(items): return sum(len(r['t'])+1 for r in items)
EPOCH_TOKENS = supervised_count(encoded['train'])
print("Target + EOS tokens per epoch:", f'{EPOCH_TOKENS:,}', '| total:',f"{EPOCH_TOKENS*CFG['epochs']:,}")
print("Example:", prompt(encoded['train'][0]['source']), repr(' '+encoded['train'][0]['target']))
ids, mask, labels = collate(encoded['train'][:2], train=True)
assert (labels[:,1:]!=-100).sum().item()==supervised_count(encoded['train'][:2])
assert (labels[mask==0]==-100).all()
assert all(labels[i,len(r['p'])].item()==r['t'][0] for i,r in enumerate(encoded['train'][:2]))
dump(OUT/'data_report.json',dict(split_sizes={s:len(encoded[s]) for s in SPLITS},split_sha256=hashes,
     target_tokens_per_epoch=EPOCH_TOKENS, target_token_budget=EPOCH_TOKENS*CFG['epochs'],
     max_sequence={s:max(len(r['p'])+len(r['t'])+1 for r in encoded[s]) for s in SPLITS}))

## 4. Models and exact token-weighted loss
LoRA uses $W=W_0+(\alpha/r)BA$ with rank 8 and alpha 16 on `c_attn` in all 12 GPT-2 blocks. GPT-2's `Conv1D` stores weights in transposed convention (`fan_in_fan_out=True`). Only 294,912 scalars should be trainable. Full fine-tuning uses all 124,439,808 scalars.

For efficiency the vocabulary projection is evaluated only at supervised positions. This is algebraically equivalent to applying masked causal-LM cross-entropy to all logits. Accumulation weights microbatches by target token count, giving the mean loss over the entire effective batch.

In [ ]:
def build_model(method):
    model=AutoModelForCausalLM.from_pretrained(MODEL_ID,revision=REVISION, torch_dtype=torch.float32)
    assert sum(p.numel() for p in model.parameters()) == 124_439_808
    model.config.pad_token_id=PAD
    if method=='lora':
        model=get_peft_model(model,LoraConfig(task_type=TaskType.CAUSAL_LM,r=CFG['lora_rank'],
            lora_alpha=CFG['lora_alpha'], lora_dropout=CFG['lora_dropout'], target_modules=['c_attn'],
            fan_in_fan_out=True,bias='none'))
        assert sum(p.numel() for p in model.parameters() if p.requires_grad)==294_912
    return model.to(DEVICE)

def base_model(model):
    return model.get_base_model() if isinstance(model,PeftModel) else model

def token_losses(model, batch):
    ids, mask, labels = (x.to(DEVICE) for x in batch)
    base = base_model(model)
    with amp_context():
        h=base.transformer(input_ids=ids, attention_mask=mask, use_cache=False, return_dict=True).last_hidden_state
        valid=labels[:,1:]!=-100
        logits=base.lm_head(h[:,:-1,:][valid])
    losses=F.cross_entropy(logits.float(),labels[:,1:][valid],reduction='none')
    return losses, valid.nonzero(as_tuple=True)[0]

@torch.inference_mode()
def reference_ce(model, examples, per_row=False):
    model.eval(); total_nll=0.; total_tokens=0; total_bytes=0; records=[]
    for b in batches(examples,CFG['eval_batch']):
        losses, indices=token_losses(model,collate(b,train=False))
        row_nll=torch.zeros(len(b),device=DEVICE).scatter_add_(0,indices,losses).cpu().tolist()
        for r,nll in zip(b,row_nll):
            nt=len(r['t']); nb=r['target_bytes']; total_nll+=nll; total_tokens+=nt; total_bytes+=nb
            if per_row: records.append(dict(id=r['id'], target_nll=nll,target_tokens=nt,target_bytes=nb))
    return dict(ce_nats_per_token=total_nll/total_tokens,bits_per_byte=total_nll/(math.log(2)*total_bytes),
                reference_tokens=total_tokens,reference_bytes=total_bytes),records

# Optional equivalence check on a tiny batch (run before training if desired).
RUN_EQUIVALENCE_CHECK=True
if RUN_EQUIVALENCE_CHECK:
    probe=build_model('full').eval()
    with torch.inference_mode():
        b=collate(encoded['validation'][:2],train=False)
        fast,_=token_losses(probe,b)
        ids,mask,lab=(x.to(DEVICE) for x in b)
        with amp_context():
            logits=probe(input_ids=ids,attention_mask=mask,use_cache=False).logits
        valid=lab[:,1:]!=-100
        standard=F.cross_entropy(logits[:,:-1,:][valid].float(),lab[:,1:][valid],reduction='none')
        torch.testing.assert_close(fast,standard,atol=0.08 if AMP else 1e-4,rtol=0.01 if AMP else 1e-4)
        print('Target-only loss equivalence check passed.')
    del probe, fast, logits, standard, ids, mask, lab, valid
    clear()

## 5. Greedy decoding and evaluation
Right padding for training, left padding for batched generation. Both methods use identical greedy decoding settings. Newline/EOS ends each sequence independently; output is cut at the first newline. SacreBLEU scores are corpus-level. Error flags are heuristic, not human annotations.

In [ ]:
BLEU_METRIC=BLEU(); CHRF_METRIC=CHRF(word_order=0,beta=2,char_order=6)
@torch.inference_mode()
def generate_rows(model, examples):
    model.eval(); results=[]
    for b in batches(examples,CFG['gen_batch']):
        prompts=[r['p'] for r in b]; width=max(map(len,prompts))
        ids=torch.full((len(b),width),PAD,dtype=torch.long,device=DEVICE); mask=torch.zeros_like(ids)
        for i,p in enumerate(prompts):
            ids[i,-len(p):]=torch.tensor(p,device=DEVICE); mask[i,-len(p):]=1
        with amp_context():
            out=model.generate(input_ids=ids,attention_mask=mask,max_new_tokens=CFG['max_new_tokens'],
                               do_sample=False,num_beams=1,use_cache=True,eos_token_id=STOP_IDS,pad_token_id=PAD)
        for r, generated in zip(b,out[:,width:].cpu().tolist()):
            stop=next((i for i,x in enumerate(generated) if x in STOP_SET),None)
            seq=generated if stop is None else generated[:stop+1]
            reason='cap' if stop is None else ('eos' if seq[-1]==EOS else 'newline')
            text=tokenizer.decode(seq,skip_special_tokens=True,clean_up_tokenization_spaces=False).split('\n',1)[0].strip()
            results.append(dict(id=r['id'],source=r['source'],reference=r['target'],prediction=text,
                                stop_reason=reason,generated_tokens=len(seq)))
    return results

def scores(records):
    p=[r['prediction'] for r in records]; ref=[r['reference'] for r in records]
    bleu=BLEU_METRIC.corpus_score(p,[ref]).score; chrf=CHRF_METRIC.corpus_score(p,[ref]).score
    return dict(bleu=bleu,chrf=chrf,bleu_signature=str(BLEU_METRIC.get_signature()),chrf_signature=str(CHRF_METRIC.get_signature()))

def evaluate(model):
    pred=generate_rows(model,encoded['test']); ce,nll=reference_ce(model,encoded['test'],per_row=True)
    frame=pd.DataFrame(pred).merge(pd.DataFrame(nll),on='id',validate='one_to_one')
    return {**scores(pred),**ce},frame

## 6. Training loop
The final checkpoint is the primary matched-budget comparison. Save the best validation-CE checkpoint for practical deployment. Training wall time includes forward/backward and optimizer steps (and data collation) but excludes validation, serialization and evaluation; peak allocated training memory is measured across training epochs, including model/optimizer allocations, excluding validation. FP16 gradient overflows are recorded. Both runs consume exactly the same supervised target tokens per epoch, though skipped optimizer updates, if any, should be disclosed.

The notebook saves completed runs and intermediate best checkpoints. Interrupted incomplete runs restart from the base checkpoint (not from optimizer state). For a single GPU, train sequentially. Add more seeds only after the first pair is working; rerunning this cell skips completed matching runs.

In [ ]:
def save_checkpoint(model,path):
    path.mkdir(parents=True,exist_ok=True)
    model.save_pretrained(path,safe_serialization=True)
    tokenizer.save_pretrained(path)

def load_checkpoint(method,path):
    if method=='lora':
        base=AutoModelForCausalLM.from_pretrained(MODEL_ID,revision=REVISION,torch_dtype=torch.float32)
        model=PeftModel.from_pretrained(base,path)
    else:
        model=AutoModelForCausalLM.from_pretrained(path,torch_dtype=torch.float32)
    model.config.pad_token_id=PAD
    return model.to(DEVICE)

def train_run(method,seed):
    run=OUT/f'{method}_seed{seed}'
    if (run/'complete.json').exists():
        print('Reusing',run); return json.loads((run/'complete.json').read_text())
    run.mkdir(parents=True,exist_ok=True)
    seed_all(seed); clear()
    model=build_model(method)
    params=[p for p in model.parameters() if p.requires_grad]
    ntrain=sum(p.numel() for p in params); nall=sum(p.numel() for p in model.parameters())
    lr=CFG[f'{method}_lr']
    # No weight decay on biases or LayerNorm vectors; same rule for both methods.
    decay=[p for p in params if p.ndim>=2]; no_decay=[p for p in params if p.ndim<2]
    optimizer=torch.optim.AdamW([dict(params=decay,weight_decay=CFG['weight_decay']),dict(params=no_decay,weight_decay=0.)],lr=lr)
    steps_per_epoch=math.ceil(math.ceil(len(encoded['train'])/CFG['micro_batch'])/CFG['accumulation'])
    total_steps=steps_per_epoch*CFG['epochs']
    scheduler=get_linear_schedule_with_warmup(optimizer,int(total_steps*CFG['warmup_fraction']),total_steps)
    scaler=torch.amp.GradScaler('cuda',enabled=AMP and AMP_DTYPE==torch.float16)
    dump(run/'config.json',dict(**experiment,gpu=GPU_NAME,method=method,seed=seed,lr=lr,trainable_parameters=ntrain,total_parameters=nall))
    val,_=reference_ce(model,encoded['validation'])
    history=[dict(epoch=0,training_tokens=0,nonpadding_tokens=0,**val,training_seconds=0.,train_nats_per_token=None)]
    best=float('inf'); best_epoch=None; total_tokens=0; total_input=0; train_seconds=0.; peak=0; skipped=0; steps=0
    for epoch in range(1,CFG['epochs']+1):
        model.train(); epoch_nll=0.; epoch_tokens=0
        if DEVICE.type=='cuda': torch.cuda.reset_peak_memory_stats()
        sync(); start=time.perf_counter()
        microbatches=epoch_batches(seed,epoch)
        groups=list(batches(microbatches,CFG['accumulation']))
        progress=tqdm(groups,desc=f'{method} seed {seed} epoch {epoch}/{CFG["epochs"]}')
        for group in progress:
            group_tokens=sum(supervised_count(b) for b in group)
            optimizer.zero_grad(set_to_none=True)
            for b in group:
                batch=collate(b,train=True)
                losses,_=token_losses(model,batch)
                nll=losses.sum()
                if not torch.isfinite(nll): raise FloatingPointError('Nonfinite training loss')
                scaler.scale(nll/group_tokens).backward()
                epoch_nll+=float(nll.detach()); epoch_tokens+=len(losses)
                total_input+=int(batch[1].sum())
                del losses,nll,batch
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(params,CFG['grad_clip'],error_if_nonfinite=not scaler.is_enabled())
            old_scale=scaler.get_scale(); scaler.step(optimizer); scaler.update()
            overflow=scaler.get_scale()<old_scale
            if overflow: skipped+=1
            else: scheduler.step()
            steps+=1; total_tokens+=group_tokens
            if steps%CFG['log_every']==0: progress.set_postfix(loss=round(epoch_nll/epoch_tokens,3),tokens=total_tokens,skipped=skipped)
        optimizer.zero_grad(set_to_none=True)
        sync(); train_seconds+=time.perf_counter()-start
        if DEVICE.type=='cuda': peak=max(peak,torch.cuda.max_memory_allocated())
        assert epoch_tokens==EPOCH_TOKENS and total_tokens==epoch*EPOCH_TOKENS
        val,_=reference_ce(model,encoded['validation'])
        history.append(dict(epoch=epoch,training_tokens=total_tokens,nonpadding_tokens=total_input,
                            **val,training_seconds=train_seconds,train_nats_per_token=epoch_nll/epoch_tokens))
        pd.DataFrame(history).to_csv(run/'learning_curve.csv',index=False)
        print('Validation bits/byte:',round(val['bits_per_byte'],4))
        if val['bits_per_byte']<best:
            best=val['bits_per_byte']; best_epoch=epoch
            save_checkpoint(model,run/'best')
    save_checkpoint(model,run/'final')
    del optimizer,scheduler,scaler,params,decay,no_decay
    clear()
    final_metrics,final_frame=evaluate(model)
    final_frame.to_csv(run/'predictions_final.csv',index=False)
    del model; clear()
    if best_epoch==CFG['epochs']:
        best_metrics,best_frame=final_metrics,final_frame
    else:
        model=load_checkpoint(method,run/'best')
        best_metrics,best_frame=evaluate(model)
        del model; clear()
    best_frame.to_csv(run/'predictions_best.csv',index=False)
    meta=dict(method=method,seed=seed,gpu=GPU_NAME,trainable_parameters=ntrain,total_parameters=nall,lr=lr,
              training_tokens=total_tokens,nonpadding_tokens=total_input,optimizer_steps_attempted=steps,
              optimizer_steps_skipped=skipped,training_seconds=train_seconds,
              peak_training_memory_mib=peak/2**20 if DEVICE.type=='cuda' else None,
              best_epoch=best_epoch,best_training_tokens=best_epoch*EPOCH_TOKENS,best_validation_bits_per_byte=best,
              final_validation_bits_per_byte=history[-1]['bits_per_byte'],final=final_metrics,best=best_metrics)
    dump(run/'complete.json',meta)
    print(method,seed,'final chrF:',round(final_metrics['chrf'],2),'best epoch:',best_epoch)
    return meta

In [ ]:
# Main experiment. Start with one paired seed; append additional seeds and rerun this cell.
run_results=[]
for seed in SEEDS:
    for method in METHODS:
        run_results.append(train_run(method,seed))
# Assert the primary comparison is at an equal training budget.
assert all(r['training_tokens']==CFG['epochs']*EPOCH_TOKENS for r in run_results)

## 7. Results and uncertainty
The primary table uses final checkpoints. An additional table reports validation-selected checkpoints for M1. Seed SD and paired sentence-bootstrap intervals answer different questions; with one seed, seed SD is missing, not zero. The paired bootstrap uses identical resampled sentence IDs for both methods and averages the corpus score difference across paired training seeds. The prediction is assessed against the observed mean delta; intervals communicate uncertainty, not a formal equivalence test.

In [ ]:
def make_table(which):
    return pd.DataFrame([{k:v for k,v in r.items() if k not in ('final','best')} | r[which] for r in run_results])
final_df=make_table('final'); best_df=make_table('best')
if final_df.get('gpu') is not None and final_df.gpu.nunique(dropna=False)>1:
    print('WARNING: runs used different GPUs', sorted(final_df.gpu.astype(str).unique()),
          '- compare training time only within one GPU; quality and memory are unaffected.')
final_df.to_csv(OUT/'results_final.csv',index=False); best_df.to_csv(OUT/'results_best.csv',index=False)
cols=['method','seed','gpu','trainable_parameters','training_tokens','bleu','chrf','bits_per_byte','training_seconds','peak_training_memory_mib','optimizer_steps_skipped']
display(final_df[cols].round(3))
metrics=['bleu','chrf','bits_per_byte','training_seconds','peak_training_memory_mib']
summary=final_df.groupby('method')[metrics].agg(['mean','std','count'])
summary.columns=['_'.join(x) for x in summary.columns]; summary.to_csv(OUT/'summary_by_method.csv')
display(summary.round(3))
print('Validation-selected M1 checkpoints:'); display(best_df[cols+['best_epoch','best_training_tokens','best_validation_bits_per_byte']].round(3))
print('BLEU:',final_df.bleu_signature.iloc[0]); print('chrF:',final_df.chrf_signature.iloc[0])

In [ ]:
# SacreBLEU 2.5.1 sufficient statistics; avoids recomputing n-grams 1,000 times.
def bootstrap_stats(metric, frame):
    return np.asarray(metric._extract_corpus_statistics(frame.prediction.fillna('').tolist(),[frame.reference.tolist()]),dtype=np.float64)
def stat_score(metric, v): return metric._compute_score_from_stats(v.tolist()).score
pairs=sorted(set(final_df.loc[final_df.method=='full','seed']) & set(final_df.loc[final_df.method=='lora','seed']))
paired=[]
if pairs:
    rng=np.random.default_rng(737)
    n=len(encoded['test'])
    counts=rng.multinomial(n,np.full(n,1/n),size=CFG['bootstrap_samples'])
    paired_metrics={}
    for name,metric in [('chrf',CHRF_METRIC),('bleu',BLEU_METRIC)]:
        diffs=[]; observed=[]
        for seed in pairs:
            full=pd.read_csv(OUT/f'full_seed{seed}'/'predictions_final.csv',dtype={'id':str},keep_default_na=False)
            lora=pd.read_csv(OUT/f'lora_seed{seed}'/'predictions_final.csv',dtype={'id':str},keep_default_na=False)
            assert full.id.tolist()==lora.id.tolist() and full.reference.tolist()==lora.reference.tolist()
            a,b=bootstrap_stats(metric,full),bootstrap_stats(metric,lora)
            av,bv=counts@a,counts@b
            diffs.append([stat_score(metric,y)-stat_score(metric,x) for x,y in zip(av,bv)])
            observed.append(stat_score(metric,b.sum(0))-stat_score(metric,a.sum(0)))
        lo,hi=np.percentile(np.mean(diffs,axis=0),[2.5,97.5])
        paired_metrics[name]=dict(delta_mean=float(np.mean(observed)),delta_seed_sd=float(np.std(observed,ddof=1)) if len(pairs)>1 else None,
                                  sentence_bootstrap_ci_low=float(lo),sentence_bootstrap_ci_high=float(hi),paired_seeds=len(pairs))
    display(pd.DataFrame(paired_metrics).T.round(3)); dump(OUT/'paired_comparison.json',paired_metrics)
    d=paired_metrics['chrf']['delta_mean']; threshold=PREDICTION['minimum_delta_chrf']
    print(f'Observed chrF delta LoRA−full: {d:.2f}; predicted >= {threshold:.2f}:',d>=threshold)
    if len(pairs)<2: print('Only one paired seed: repeat before claiming robustness across training seeds.')
else: print('Need both methods at a common seed for paired comparison.')

In [ ]:
# Report-style figures: same look as the M0 figures in the report (STIX serif, 6.5/7 pt text,
# 1 pt open axes, faint grid, tab10 colours). One panel = one 0.24\textwidth subfigure.
# Each figure is saved as PDF (for LaTeX) and PNG. Thin lines / light points are individual seeds;
# bands and error bars are ± 1 SD across seeds.
from matplotlib.ticker import MaxNLocator
plt.rcdefaults()
plt.rcParams.update({
    'font.family': 'STIXGeneral', 'mathtext.fontset': 'stix', 'pdf.fonttype': 42, 'ps.fonttype': 42,
    'font.size': 6.5, 'axes.labelsize': 7, 'xtick.labelsize': 6.5, 'ytick.labelsize': 6.5,
    'legend.fontsize': 6, 'legend.title_fontsize': 6, 'legend.frameon': False,
    'legend.handletextpad': .4, 'legend.borderaxespad': .2, 'legend.labelspacing': .3,
    'axes.linewidth': 1, 'axes.spines.top': False, 'axes.spines.right': False,
    'axes.grid': True, 'grid.alpha': .25, 'grid.linewidth': .8,
    'xtick.major.width': 1, 'ytick.major.width': 1, 'xtick.major.size': 3, 'ytick.major.size': 3,
    'lines.linewidth': 1, 'lines.markersize': 3, 'errorbar.capsize': 1.5,
    'figure.dpi': 200, 'savefig.dpi': 300,
})
PANEL = (1.62, 1.40)  # inches; one quarter-width subfigure in the report
colors = {'full': 'tab:blue', 'lora': 'tab:orange'}
markers = {'full': 'o', 'lora': 's'}
names = {'full': 'full fine-tuning', 'lora': f'LoRA (rank {CFG["lora_rank"]})'}

def panels(n=1):
    fig, axes = plt.subplots(1, n, figsize=(PANEL[0] * n, PANEL[1]), squeeze=False)
    for ax in axes[0]:
        for side in ('left', 'bottom'): ax.spines[side].set_position(('outward', 3))
    return fig, axes[0]

def finish(fig, name):
    fig.savefig(OUT / f'{name}.pdf', bbox_inches='tight', pad_inches=.02)
    fig.savefig(OUT / f'{name}.png', bbox_inches='tight', pad_inches=.02)
    plt.show(); plt.close(fig)

# (1) Q1 learning curves: validation CE against supervised target tokens (report Figure: Q1 training).
fig, (ax,) = panels()
for method in METHODS:
    curves = [pd.read_csv(OUT / f'{method}_seed{r["seed"]}' / 'learning_curve.csv')
              for r in run_results if r['method'] == method]
    if not curves: continue
    x = curves[0].training_tokens / 1e6
    vals = np.stack([c.bits_per_byte.to_numpy() for c in curves])
    for c in curves: ax.plot(c.training_tokens / 1e6, c.bits_per_byte, color=colors[method], alpha=.25, lw=.7)
    if len(curves) > 1:
        sd = vals.std(0, ddof=1)
        ax.fill_between(x, vals.mean(0) - sd, vals.mean(0) + sd, color=colors[method], alpha=.15, lw=0)
    ax.plot(x, vals.mean(0), color=colors[method], marker=markers[method], label=names[method])
ax.xaxis.set_major_locator(MaxNLocator(4))
ax.set(xlabel='Target tokens (millions)', ylabel='Validation CE (bits/byte)')
ax.legend(loc='upper right'); finish(fig, 'q1_learning_curves')

# (2) Final-checkpoint test quality: one panel per metric, seeds as light points, mean ± SD.
def dot_panels(specs, name):
    fig, axes = panels(len(specs))
    for ax, (key, label) in zip(axes, specs):
        for i, method in enumerate(METHODS):
            v = final_df.loc[final_df.method == method, key].dropna().to_numpy(dtype=float)
            if not len(v): continue
            ax.scatter(np.full(len(v), i) + np.linspace(-.08, .08, len(v)), v, s=8, lw=0,
                       color=colors[method], alpha=.45, marker=markers[method])
            ax.errorbar(i + .22, v.mean(), yerr=v.std(ddof=1) if len(v) > 1 else None, fmt=markers[method],
                        color=colors[method], ms=3.5, elinewidth=.9)
        ax.set_xticks(range(len(METHODS)), ['full' if m == 'full' else 'LoRA' for m in METHODS])
        ax.set_xlim(-.5, len(METHODS) - .5); ax.grid(axis='x', visible=False); ax.set_ylabel(label)
        ax.yaxis.set_major_locator(MaxNLocator(4))
    fig.tight_layout(w_pad=1.4); finish(fig, name)

dot_panels([('chrf', 'Test chrF'), ('bleu', 'Test BLEU'), ('bits_per_byte', 'Test CE (bits/byte)')],
           'q1_final_quality')
# (3) Training cost at the same token budget.
dot_panels([('training_seconds', 'Training time (s)'), ('peak_training_memory_mib', 'Peak memory (MiB)')],
           'q1_training_cost')
print('Saved report figures (PDF + PNG) to', OUT)

## 8. Failure review and working demonstration
The following flags approximate the report’s automatic checks. They are screening tools; inspect the exported review sheet and write manual labels. Choose the demonstration checkpoint by validation CE only, not test BLEU/chrF. The demonstration translates new sentences that appear in no split (T5). For Notebook 2, export a regular merged GPT-2 checkpoint regardless of the winning method.

In [ ]:
DE=set('der die das den dem des ein eine einer eines einem einen und ist sind war waren nicht ich wir sie er es zu mit von auf für als auch sich im in'.split())
EN=set('the a an and is are was were not i we you she he it to with of on for as also in'.split())
def words(s): return re.findall(r'\b\w+\b',s.lower())
def inspect_frame(frame):
    records=[]
    for r in frame.to_dict('records'):
        p,s,t=r['prediction'],r['source'],r['reference']; w=words(p)
        german=sum(x in DE for x in w); english=sum(x in EN for x in w)
        digits=lambda x:re.findall(r'\d+',x)
        shared_names=set(re.findall(r'\b[A-Z][a-z]+\b',s)) & set(re.findall(r'\b[A-Z][a-z]+\b',t))
        records.append(dict(**r,copy_flag=(p.strip()==s.strip() or CHRF_METRIC.sentence_score(p,[s]).score>=60),
            german_flag=german>=2 and german>english,omission_flag=len(words(t))>=4 and len(w)<len(words(t))/2,
            name_number_flag=digits(p)!=digits(t) or not shared_names.issubset(set(re.findall(r'\b[A-Z][a-z]+\b',p))),
            cap_flag=r['stop_reason']=='cap',sentence_chrf=CHRF_METRIC.sentence_score(p,[t]).score,manual_label=''))
    return pd.DataFrame(records)
error_rows=[]
for r in run_results:
    method,seed=r['method'],r['seed']; path=OUT/f'{method}_seed{seed}'
    f=inspect_frame(pd.read_csv(path/'predictions_best.csv',dtype={'id':str},keep_default_na=False))
    f.to_csv(path/'error_review_all.csv',index=False)
    # Seeded manual sample plus low-scoring examples; clearly distinguish selection criteria.
    review=pd.concat([f.sample(min(12,len(f)),random_state=737).assign(review_selection='random'),
                      f.nsmallest(min(8,len(f)),'sentence_chrf').assign(review_selection='lowest chrF')]).drop_duplicates('id')
    review.to_csv(path/'error_review_sample.csv',index=False)
    rates={k:100*f[k].mean() for k in ['copy_flag','german_flag','omission_flag','name_number_flag','cap_flag']}
    error_rows.append(dict(method=method,seed=seed,**rates,top3_pct=100*f.prediction.value_counts().head(3).sum()/len(f)))
error_df=pd.DataFrame(error_rows); error_df.to_csv(OUT/'error_rates.csv',index=False); display(error_df.round(2))
chosen=min(run_results,key=lambda r:r['best_validation_bits_per_byte'])
chosen_path=OUT/f"{chosen['method']}_seed{chosen['seed']}"/'best'
print('Selected on validation:',chosen['method'],chosen['seed'],'epoch',chosen['best_epoch'])
display(pd.read_csv(chosen_path.parent/'error_review_sample.csv',keep_default_na=False)[['source','reference','prediction','sentence_chrf','review_selection']])

In [ ]:
# Export a standalone GPT-2 model (merged LoRA if necessary) for the cache benchmark.
EXPORT=OUT/'best_model_for_inference'
if 'demo_model' in globals(): del demo_model
clear()
demo_model=load_checkpoint(chosen['method'],chosen_path)
if isinstance(demo_model,PeftModel): demo_model=demo_model.merge_and_unload()
demo_model.eval()
save_checkpoint(demo_model,EXPORT)
dump(OUT/'selected_model.json',dict(method=chosen['method'],seed=chosen['seed'],epoch=chosen['best_epoch'],
     path=str(EXPORT.resolve()),validation_bits_per_byte=chosen['best_validation_bits_per_byte'],
     prompt_template='German: {source}\nEnglish:',max_new_tokens=CFG['max_new_tokens'],base_revision=REVISION))
print('Notebook 2 checkpoint:', EXPORT.resolve())

In [ ]:
# Working demonstration (T5): new sentences that appear in no split. Edit or add your own.
NEW_SOURCES=['Morgen fahre ich mit dem Zug nach Hamburg, um meine Großmutter zu besuchen.',
             'Die Bibliothek ist am Sonntag geschlossen.',
             'Wir haben drei Jahre lang an diesem Projekt gearbeitet.',
             'Kannst du mir sagen, wo der nächste Bahnhof ist?']
known={r['source'].strip() for s in SPLITS for r in rows[s]}
assert not any(s.strip() in known for s in NEW_SOURCES), 'Demo sentences must not come from the shared splits.'
demo_rows=[]
for i,source in enumerate(NEW_SOURCES):
    p=encode_text(prompt(source))
    assert len(p)+CFG['max_new_tokens']<=CFG['context']
    demo_rows.append(dict(id=f'demo-{i}',source=source,target='',p=p))
for r in generate_rows(demo_model,demo_rows):
    print('DE:',r['source']); print('EN:',r['prediction']); print()

## 9. Export results
The ZIP includes configuration, preregistration, data fingerprints, learning curves, metrics, predictions, manual-review sheets, and figures. Model weights stay in the experiment folder (prefer Drive or download separately). For the report, use `results_final.csv` for matched-budget Q1(a), `results_best.csv` for validation-selected M1 and `paired_comparison.json` for sentence uncertainty. Do not report smoke scores.

In [ ]:
archive=OUT.parent/f'{OUT.name}_report.zip'
with zipfile.ZipFile(archive,'w',compression=zipfile.ZIP_DEFLATED) as z:
    for p in OUT.rglob('*'):
        if p.is_file() and p.suffix in {'.csv','.json','.png'} and not any(part in {'best','final','best_model_for_inference'} for part in p.relative_to(OUT).parts):
            z.write(p,p.relative_to(OUT))
print('Report archive:',archive.resolve())
print('Keep this checkpoint for Notebook 2:',EXPORT.resolve())
# Optional: from google.colab import files; files.download(str(archive))